# Evaluate Fine-tuned SAM2 on Robot Arm Dataset

Validates the fine-tuned model on 10 held-out val scenes (scene_0041 ~ scene_0050).

**VSCode Remote Jupyter** — plots render inline directly.

### Evaluation protocol
- Prompt: GT mask of **frame 0** for both arm (obj_id=1) and gripper (obj_id=2)
- Propagate forward through all remaining frames
- Metric: mean IoU (J-score) per object per scene

In [ ]:
# Cell 1 — Imports & Paths
%matplotlib inline
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch

# ── Adjust this to your actual SAM2 repo path ──────────────────────────────
SAM2_ROOT = "/path/to/sam2_repo"   # e.g. "/home/haoxiang/sam2"
sys.path.insert(0, SAM2_ROOT)

# ── Paths ──────────────────────────────────────────────────────────────────
CKPT_PATH  = "/data/haoxiang/logs/sam2_finetune/airexo_task0013/checkpoints/checkpoint.pt"
IMG_ROOT   = "/data/haoxiang/data/airexo2_processed/sam2_finetune/JPEGImages"
ANN_ROOT   = "/data/haoxiang/data/airexo2_processed/sam2_finetune/Annotations"
VAL_SCENES = [f"scene_{i:04d}" for i in range(41, 51)]

print(f"Val scenes: {VAL_SCENES}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")

In [ ]:
# Cell 2 — Load fine-tuned model
# NOTE: use the model-architecture config (sam2.1/), NOT the training config (sam2.1_training/)
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(
    config_file="configs/sam2.1/sam2.1_hiera_b+.yaml",
    ckpt_path=CKPT_PATH,
    device="cuda",
    mode="eval",
)
print("Model loaded successfully.")

In [ ]:
# Cell 3 — Helper functions

def load_binary_mask(ann_root, scene, obj_folder, frame_idx):
    """Load a binary mask from Annotations/{scene}/{obj_folder}/{frame_idx:05d}.png.
    Returns np.ndarray of shape (H, W), dtype bool. White pixel (255) = foreground."""
    path = os.path.join(ann_root, scene, obj_folder, f"{frame_idx:05d}.png")
    arr = np.array(Image.open(path).convert("L"))
    return arr > 128


def load_rgb_frame(img_root, scene, frame_idx):
    """Load an RGB frame from JPEGImages/{scene}/{frame_idx:05d}.jpg."""
    path = os.path.join(img_root, scene, f"{frame_idx:05d}.jpg")
    return np.array(Image.open(path).convert("RGB"))


def overlay_mask(image, mask, color, alpha=0.45):
    """Overlay a boolean mask on an RGB image with a given RGBA color."""
    out = image.copy().astype(float)
    for c, v in enumerate(color[:3]):
        out[..., c] = np.where(mask, out[..., c] * (1 - alpha) + v * 255 * alpha, out[..., c])
    return out.clip(0, 255).astype(np.uint8)


def compute_iou(pred, gt):
    """Compute IoU between two boolean masks. Returns 1.0 if both are empty."""
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return float(inter) / float(union) if union > 0 else 1.0


def count_frames(img_root, scene):
    return len(sorted(glob.glob(os.path.join(img_root, scene, "*.jpg"))))


print("Helper functions defined.")

In [ ]:
# Cell 4 — Single-scene visualization (quick sanity check)

VIS_SCENE  = "scene_0041"
VIS_FRAMES = [0, 50, 100, 150]   # adjust if scene has fewer frames

video_dir = os.path.join(IMG_ROOT, VIS_SCENE)
n_frames  = count_frames(IMG_ROOT, VIS_SCENE)
VIS_FRAMES = [f for f in VIS_FRAMES if f < n_frames]
print(f"{VIS_SCENE}: {n_frames} frames, visualising {VIS_FRAMES}")

# --- Propagate ---
pred_masks = {}
with torch.inference_mode():
    state = predictor.init_state(video_path=video_dir)
    predictor.reset_state(state)

    arm_mask_0 = load_binary_mask(ANN_ROOT, VIS_SCENE, "0", 0)
    grp_mask_0 = load_binary_mask(ANN_ROOT, VIS_SCENE, "1", 0)
    predictor.add_new_mask(state, frame_idx=0, obj_id=1, mask=arm_mask_0)
    predictor.add_new_mask(state, frame_idx=0, obj_id=2, mask=grp_mask_0)

    for f_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
        pred_masks[f_idx] = {
            oid: (mask_logits[i].squeeze().cpu().numpy() > 0.0)
            for i, oid in enumerate(obj_ids)
        }

# --- Plot ---
ARM_COLOR = (0.0, 0.6, 1.0)   # blue
GRP_COLOR = (1.0, 0.3, 0.0)   # orange
n_cols = len(VIS_FRAMES)

fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 12))
if n_cols == 1:
    axes = axes[:, np.newaxis]

for col, f_idx in enumerate(VIS_FRAMES):
    img = load_rgb_frame(IMG_ROOT, VIS_SCENE, f_idx)
    arm_pred = pred_masks.get(f_idx, {}).get(1, np.zeros(img.shape[:2], bool))
    grp_pred = pred_masks.get(f_idx, {}).get(2, np.zeros(img.shape[:2], bool))

    axes[0, col].imshow(img);                             axes[0, col].set_title(f"Frame {f_idx}")
    axes[1, col].imshow(overlay_mask(img, arm_pred, ARM_COLOR)); axes[1, col].set_title(f"arm (pred)")
    axes[2, col].imshow(overlay_mask(img, grp_pred, GRP_COLOR)); axes[2, col].set_title(f"gripper (pred)")

for ax in axes.flatten():
    ax.axis("off")

legend = [mpatches.Patch(color=ARM_COLOR, label="arm"), mpatches.Patch(color=GRP_COLOR, label="gripper")]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=12)
plt.suptitle(f"{VIS_SCENE} — fine-tuned SAM2", fontsize=14)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
plt.show()

In [ ]:
# Cell 5 — Quantitative evaluation: IoU on all 10 val scenes

results = {}   # scene -> {"arm": mean_iou, "gripper": mean_iou}

for scene in VAL_SCENES:
    video_dir = os.path.join(IMG_ROOT, scene)
    n_frames  = count_frames(IMG_ROOT, scene)

    arm_ious, grp_ious = [], []

    with torch.inference_mode():
        state = predictor.init_state(video_path=video_dir)
        predictor.reset_state(state)

        predictor.add_new_mask(state, 0, 1, load_binary_mask(ANN_ROOT, scene, "0", 0))
        predictor.add_new_mask(state, 0, 2, load_binary_mask(ANN_ROOT, scene, "1", 0))

        for f_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
            pred = {
                oid: (mask_logits[i].squeeze().cpu().numpy() > 0.0)
                for i, oid in enumerate(obj_ids)
            }
            # Load GT masks (may not exist for every frame)
            try:
                gt_arm = load_binary_mask(ANN_ROOT, scene, "0", f_idx)
                gt_grp = load_binary_mask(ANN_ROOT, scene, "1", f_idx)
            except FileNotFoundError:
                continue

            arm_ious.append(compute_iou(pred.get(1, np.zeros_like(gt_arm)), gt_arm))
            grp_ious.append(compute_iou(pred.get(2, np.zeros_like(gt_grp)), gt_grp))

    results[scene] = {
        "arm":     float(np.mean(arm_ious))     if arm_ious else float("nan"),
        "gripper": float(np.mean(grp_ious))     if grp_ious else float("nan"),
        "n_frames": len(arm_ious),
    }
    print(f"{scene} ({results[scene]['n_frames']} frames): "
          f"arm J={results[scene]['arm']:.3f}  gripper J={results[scene]['gripper']:.3f}")

# Summary
all_arm = np.nanmean([v["arm"]     for v in results.values()])
all_grp = np.nanmean([v["gripper"] for v in results.values()])
print(f"\n{'='*50}")
print(f"Mean arm J     : {all_arm:.3f}")
print(f"Mean gripper J : {all_grp:.3f}")
print(f"Mean J&F (avg) : {(all_arm + all_grp) / 2:.3f}")

In [ ]:
# Cell 6 — Bar chart summary across all val scenes

scenes   = list(results.keys())
arm_vals = [results[s]["arm"]     for s in scenes]
grp_vals = [results[s]["gripper"] for s in scenes]

x     = np.arange(len(scenes))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, arm_vals, width, label="arm",     color="steelblue")
ax.bar(x + width/2, grp_vals, width, label="gripper", color="coral")
ax.axhline(all_arm, color="steelblue", linestyle="--", linewidth=1, label=f"arm mean={all_arm:.3f}")
ax.axhline(all_grp, color="coral",     linestyle="--", linewidth=1, label=f"gripper mean={all_grp:.3f}")

ax.set_xlabel("Scene")
ax.set_ylabel("IoU (J-score)")
ax.set_title("Per-scene IoU — fine-tuned SAM2 (val set)")
ax.set_xticks(x)
ax.set_xticklabels([s.replace("scene_", "") for s in scenes], rotation=30)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 (optional) — Per-frame IoU curve for one scene
# Detects tracking drift: if IoU drops significantly over time, the model loses track.

CURVE_SCENE = "scene_0041"
video_dir   = os.path.join(IMG_ROOT, CURVE_SCENE)

frame_arm_ious, frame_grp_ious, frame_ids = [], [], []

with torch.inference_mode():
    state = predictor.init_state(video_path=video_dir)
    predictor.reset_state(state)
    predictor.add_new_mask(state, 0, 1, load_binary_mask(ANN_ROOT, CURVE_SCENE, "0", 0))
    predictor.add_new_mask(state, 0, 2, load_binary_mask(ANN_ROOT, CURVE_SCENE, "1", 0))

    for f_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
        pred = {oid: (mask_logits[i].squeeze().cpu().numpy() > 0.0) for i, oid in enumerate(obj_ids)}
        try:
            gt_arm = load_binary_mask(ANN_ROOT, CURVE_SCENE, "0", f_idx)
            gt_grp = load_binary_mask(ANN_ROOT, CURVE_SCENE, "1", f_idx)
        except FileNotFoundError:
            continue
        frame_ids.append(f_idx)
        frame_arm_ious.append(compute_iou(pred.get(1, np.zeros_like(gt_arm)), gt_arm))
        frame_grp_ious.append(compute_iou(pred.get(2, np.zeros_like(gt_grp)), gt_grp))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(frame_ids, frame_arm_ious, label="arm",     color="steelblue")
ax.plot(frame_ids, frame_grp_ious, label="gripper", color="coral")
ax.set_xlabel("Frame index")
ax.set_ylabel("IoU")
ax.set_title(f"{CURVE_SCENE} — per-frame IoU over time")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()